# Tutorial 0: Project setup and first steps with PyTorch

This tutorial is not graded. It makes sure that your computer is ready for the next tutorials.

# Your Task

1. Install VSCode and Python if these are not on your computer
2. Create a virtual environment for the course
3. Install PyTorch and the packages listed in `requirements.txt` in this environment
4. Test the installation
5. Take your first steps with PyTorch tensors and automatic differentiation

## Installing the tools

### VSCode
1. Install [VSCode](https://code.visualstudio.com/)
2. In VSCode, open the Extensions view (Ctrl+Shift+X) and install the **Python** and **Jupyter** extensions (both from Microsoft)
3. Open the course folder via

    File > Open Folder

4. Get familiar with the GUI and the icons on the left vertical bar

### Python
You need Python **3.10 or newer**.
- **Windows**: install Python from [python.org](https://www.python.org/downloads/windows/) (tick _"Add python.exe to PATH"_), or run `winget install Python.Python.3.12` in a terminal. Then use the `py` launcher in the terminal: `py --version`.

    If typing `python` opens the Microsoft Store, this is a Windows alias: disable it in _Settings > Apps > Advanced app settings > App execution aliases_, or use `py` instead of `python`.
- **macOS / Linux**: Python is usually installed. Check with `python3 --version`.

## Creating and managing a virtual environment

A virtual environment is a folder containing a Python interpreter and its own set of libraries. Each project can then use its own library versions without interfering with the others.

### From a terminal
1. Open a terminal (in VSCode: Terminal > New Terminal)
2. Place yourself in the course directory

```
cd path/to/the/course/directory
```
3. Create a virtual environment in a subdirectory called `.venv`

```
py -m venv .venv          (Windows)
python3 -m venv .venv     (macOS / Linux)
```

If you don't need this environment anymore, you can simply delete the `.venv` folder.

### From VSCode
1. Open the course folder
2. Open the command palette (Ctrl+Shift+P)
3. Look for _"Python: Create Environment"_
4. Select _Venv_, then the Python interpreter for this environment
5. If VSCode proposes to install the packages from `requirements.txt`, you may let it install them

### Activating an environment
- Windows (PowerShell)
```
.venv\Scripts\Activate.ps1
```
If PowerShell refuses to run the script, run once `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`.

- Windows (cmd)
```
.venv\Scripts\activate.bat
```
- macOS / Linux
```
source .venv/bin/activate
```

The name of the environment, `(.venv)`, now appears at the beginning of the terminal prompt. If you run `pip` to install or manage Python modules, these will be installed in the `.venv` directory.

To deactivate the environment, simply type in the terminal
```
deactivate
```

## Installing the libraries

You can install libraries on the fly, e.g.
```
pip install numpy
```

For a given project, the libraries are usually listed in a text file (e.g. `requirements.txt`). Install all the libraries of the course, with the environment activated, by running from the course directory:
```
pip install -r requirements.txt
```

### PyTorch and the GPU
The command above installs [PyTorch](https://pytorch.org/).
- **Windows and Linux without NVIDIA GPU, macOS**: nothing else to do. On Macs with Apple Silicon, PyTorch can use the GPU through the `mps` backend.
- **Windows and Linux with an NVIDIA GPU**: on Windows, the default PyTorch package only uses the CPU. Install the CUDA version of PyTorch **before** the other libraries:
```
pip install torch --index-url https://download.pytorch.org/whl/cu126
pip install -r requirements.txt
```
The [PyTorch website](https://pytorch.org/get-started/locally/) gives the exact command for your system. You only need an up-to-date NVIDIA driver: the CUDA libraries are included in the PyTorch package.

A GPU is **not required** for this course: all the tutorials run on a laptop CPU.

### Note for VSCode
When running a Jupyter notebook for the first time, VSCode asks you to select a kernel. Select the `.venv` environment you created (_"Python Environments"_ > `.venv`).

**Task 1**
- Run the cell below. It should print the version of PyTorch and the device available for computations.

In [1]:
# control the PyTorch installation on your computer
import sys
import torch

print('Python ', sys.version.split()[0])
print('PyTorch', torch.__version__)
print('CUDA GPU available:', torch.cuda.is_available())
print('Apple GPU (mps) available:', torch.backends.mps.is_available())

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print('Device used in the tutorials:', device)

Python  3.12.10
PyTorch 2.14.1+cpu
CUDA GPU available: False
Apple GPU (mps) available: False
Device used in the tutorials: cpu


In [2]:
# control the other libraries of the course
import numpy, scipy, pandas, sklearn, matplotlib, wfdb, h5py

for module in (numpy, scipy, pandas, sklearn, matplotlib, wfdb, h5py):
    print(f'{module.__name__:12s} {module.__version__}')

numpy        2.5.4
scipy        1.18.1
pandas       3.0.6
sklearn      1.9.1
matplotlib   3.11.2
wfdb         4.3.1
h5py         3.16.0


# (Optional) Managing the environment with conda

Instead of `venv`, you may use conda to manage your environments. We recommend [Miniforge](https://github.com/conda-forge/miniforge), a minimal conda installer that uses the community channel _conda-forge_.

1. Create the environment described in the file `condaenv.yml` of this tutorial (Python from conda, then all the course libraries from `requirements.txt` via pip)
```
conda env create -f condaenv.yml
```
2. List the environments available on your machine
```
conda env list
```
3. Activate the environment (`dl4biosig` is the name given in `condaenv.yml`)
```
conda activate dl4biosig
```
4. For an NVIDIA GPU, reinstall the CUDA version of PyTorch in the activated environment, as described above.
5. Deactivate the environment
```
conda deactivate
```
6. Remove the environment
```
conda env remove -n dl4biosig
```

# First steps with PyTorch

## Tensors

PyTorch computes on **tensors** (`torch.Tensor`): multidimensional arrays similar to NumPy arrays, which can additionally live on a GPU and record the operations applied to them to compute gradients.

In [3]:
import numpy as np

a = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
print(a)
print('shape:', a.shape, '| dtype:', a.dtype, '| device:', a.device)

tensor([[1., 2., 3.],
        [4., 5., 6.]])
shape: torch.Size([2, 3]) | dtype: torch.float32 | device: cpu


**Task 2**
- Create a NumPy array `x_np` containing 5 values regularly spaced between 0 and 1, convert it into a tensor `x` with `torch.from_numpy`, and print its dtype.
- Convert `x` into a `float32` tensor (`.float()`), compute `y = 2 * x + 1`, and convert `y` back into a NumPy array.
- Compute the matrix product of `a` with its transpose (`a @ a.T`). What is the shape of the result?

In [8]:
# 1
x_np = np.linspace(0, 1, 5)
x = torch.from_numpy(x_np)
print('x dtype:', x.dtype, '\nx_np:', x_np)

# 2
x = x.float()
y = 2 * x + 1
y_np = y.numpy()
print('y_np: ', y_np)

prod = a @ a.T
print('prod shape', prod.shape)

x dtype: torch.float64 
x_np: [0.   0.25 0.5  0.75 1.  ]
y_np:  [1.  1.5 2.  2.5 3. ]
prod shape torch.Size([2, 2])


**Answer**

Square matrix 2x2

**Task 3**
- Move the tensor `a` to the device found in Task 1 with `.to(device)`, and print its device.
- Tensors involved in an operation must be on the same device. What happens if you compute `a.to(device) + a` on a computer with a GPU?

In [17]:
a.to(device)
print('Device of a:', a.device)


Device of a: cpu


**Answer**

If the computer has a GPU, then the device = 'cuda'.  
But the a itself is in the RAM of CPU.  
So the operation a.to(device) + a will bring an error. While my computer is not, as it has no GPU:)

## Automatic differentiation

When a tensor is created with `requires_grad=True`, PyTorch records the operations applied to it. Calling `.backward()` on a scalar result then computes its gradient with respect to this tensor, stored in `.grad`. This is how neural networks are trained.

**Task 4**
- Consider $f(x) = x^2 + 2x$. Compute by hand the derivative $f'(x)$ and its value at $x = 3$.
- Check your result with PyTorch: create `x = torch.tensor(3., requires_grad=True)`, compute `f = x**2 + 2*x`, call `f.backward()` and print `x.grad`.

**Answer**

Your answer

## A linear regression by gradient descent

We generate noisy data following $y = 2x - 1$ and try to recover the slope $w = 2$ and the intercept $b = -1$ by gradient descent on the mean squared error

$$L(w, b) = \frac{1}{n}\sum_{i=1}^n (w x_i + b - y_i)^2$$

In [ ]:
torch.manual_seed(0)
x = torch.rand(100, 1)
y = 2 * x - 1 + 0.1 * torch.randn(100, 1)

import matplotlib.pyplot as plt
plt.scatter(x, y, s=10)
plt.xlabel('x')
plt.ylabel('y')
plt.show()

**Task 5**

Complete the loop below. At each step:
1. compute the predictions `y_pred = w * x + b` and the loss `loss` (mean squared error)
2. compute the gradients with `loss.backward()`
3. update the parameters: $w \leftarrow w - \eta \frac{\partial L}{\partial w}$ and $b \leftarrow b - \eta \frac{\partial L}{\partial b}$, inside a `with torch.no_grad():` block (the update itself must not be recorded)
4. reset the gradients to zero with `w.grad.zero_()` and `b.grad.zero_()` (PyTorch accumulates gradients)

Do you recover $w \approx 2$ and $b \approx -1$?

In [ ]:
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.5  # learning rate (eta)

for step in range(200):
    pass  # Your code
    if step % 50 == 0:
        print(f'step {step:3d} - loss {loss.item():.4f} - w {w.item():.3f} - b {b.item():.3f}')

print(f'final: w = {w.item():.3f}, b = {b.item():.3f}')

PyTorch provides building blocks for all these steps: `nn.Linear` holds the parameters $w$ and $b$, `nn.MSELoss` computes the loss, and an optimizer from `torch.optim` performs the update and resets the gradients. The same regression then reads as below. You will find exactly this structure in the training loop of Case Study 1.

In [ ]:
from torch import nn

model = nn.Linear(1, 1)  # y = w x + b
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for step in range(200):
    optimizer.zero_grad()        # reset the gradients
    loss = loss_fn(model(x), y)  # forward pass and loss
    loss.backward()              # gradients
    optimizer.step()             # update of the parameters

print(f'final: w = {model.weight.item():.3f}, b = {model.bias.item():.3f}')